In [1]:
import pandas as pd
import numpy as np

In [2]:
df1 = pd.read_csv("../data/Year 2009-2010.csv")
df2 = pd.read_csv("../data/Year 2010-2011.csv")

In [49]:
print(df1.shape)
print(df2.shape)

df1.head()

(525461, 12)
(541910, 13)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Customer_Segment,Sales_Channel,Payment_Mode,Order_Priority
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Regular,Online,UPI,High
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Regular,Online,UPI,Low
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Premium,Offline,Card,Medium
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,New,Retail,Cash,Low
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Regular,Online,Net Banking,High


In [50]:
df1 = df1.loc[:, ~df1.columns.str.contains("^Unnamed")]
df2 = df2.loc[:, ~df2.columns.str.contains("^Unnamed")]

print(df1.columns)

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country', 'Customer_Segment', 'Sales_Channel',
       'Payment_Mode', 'Order_Priority'],
      dtype='object')


In [51]:
print(df1.isnull().sum())

Invoice                  0
StockCode                0
Description           2928
Quantity                 0
InvoiceDate              0
Price                    0
Customer ID         107927
Country                  0
Customer_Segment         0
Sales_Channel            0
Payment_Mode             0
Order_Priority           0
dtype: int64


In [4]:
print("Duplicate Rows:", df1.duplicated().sum())

Duplicate Rows: 835


In [5]:
df1 = df1.drop_duplicates()

print("Remaining Rows:", len(df1))

Remaining Rows: 524626


In [6]:
df1["Description"] = df1["Description"].fillna("Unknown")

In [7]:
print(df1["Description"].isnull().sum())

0


In [52]:
df1["InvoiceDate"] = pd.to_datetime(
    df1["InvoiceDate"],
    errors="coerce"
)

In [9]:
df1 = df1.dropna(subset=["InvoiceDate"])

print("InvoiceDate Nulls:", df1["InvoiceDate"].isnull().sum())

InvoiceDate Nulls: 0


In [53]:
df1["SalesAmount"] = df1["Quantity"] * df1["Price"]

df1[["Quantity","Price","SalesAmount"]].head()

,Quantity,Price,SalesAmount
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [8]:
df1 = df1.dropna(subset=["Customer ID"])

print("Customer ID Nulls:", df1["Customer ID"].isnull().sum())

Customer ID Nulls: 0


In [54]:
df1 = df1.dropna(subset=["Customer ID"])

print(df1.shape)

(417534, 13)


In [10]:
print("Negative Quantity Rows:",
      (df1["Quantity"] < 0).sum())

Negative Quantity Rows: 9836


In [12]:
df1 = df1[df1["Quantity"] >= 0]

In [13]:
print("Negative Price Rows:",
      (df1["Price"] < 0).sum())

Negative Price Rows: 0


In [14]:
df1 = df1[df1["Price"] >= 0]

In [15]:
print("Missing Values:")
print(df1.isnull().sum())

print("\nDuplicate Rows:")
print(df1.duplicated().sum())

print("\nNegative Quantity:")
print((df1["Quantity"] < 0).sum())

print("\nNegative Price:")
print((df1["Price"] < 0).sum())

Missing Values:
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID         0
Country             0
Customer_Segment    0
Sales_Channel       0
Payment_Mode        0
Order_Priority      0
dtype: int64

Duplicate Rows:
0

Negative Quantity:
0

Negative Price:
0


In [16]:
df1.to_csv("Year_2009_2010_Cleaned.csv", index=False)
df2.to_csv("Year_2010_2011_Cleaned.csv", index=False)

print("Both cleaned files saved successfully!")

Both cleaned files saved successfully!


In [55]:
snapshot_date = df1["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df1.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "Invoice": "nunique",
    "SalesAmount": "sum"
})

rfm.columns = ["Recency","Frequency","Monetary"]

rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,67,15,-64.68
12347.0,3,2,1323.32
12348.0,74,1,222.16
12349.0,43,4,2646.99
12351.0,11,1,300.93


In [56]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5,4,3,2,1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1,2,3,4,5]
)

In [57]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
Customer ID,,,,,,,
12346.0,67,15,-64.68,3,5,1,351
12347.0,3,2,1323.32,5,2,4,524
12348.0,74,1,222.16,2,1,1,211
12349.0,43,4,2646.99,3,3,5,335
12351.0,11,1,300.93,5,1,2,512


In [58]:
rfm.to_csv("rfm_features.csv")

print("RFM Features Saved")

RFM Features Saved


In [59]:
daily_sales = (
    df1.groupby(df1["InvoiceDate"].dt.date)["SalesAmount"]
    .sum()
    .reset_index()
)

daily_sales.head()

,InvoiceDate,SalesAmount
0,2009-12-01,42708.22
1,2009-12-02,52578.19
2,2009-12-03,61534.22
3,2009-12-04,33686.86
4,2009-12-05,9803.05


In [60]:
daily_sales["RollingMean7"] = (
    daily_sales["SalesAmount"]
    .rolling(window=7)
    .mean()
)

daily_sales.head(10)

,InvoiceDate,SalesAmount,RollingMean7
0,2009-12-01,42708.22,NaN
1,2009-12-02,52578.19,NaN
2,2009-12-03,61534.22,NaN
3,2009-12-04,33686.86,NaN
4,2009-12-05,9803.05,NaN
5,2009-12-06,24284.28,NaN
6,2009-12-07,32423.30,36716.874286
7,2009-12-08,39030.25,36191.450000
8,2009-12-09,30894.06,33093.717143
9,2009-12-10,37706.73,29689.790000


In [61]:
daily_sales["RollingStd7"] = (
    daily_sales["SalesAmount"]
    .rolling(window=7)
    .std()
)

daily_sales.head(10)

,InvoiceDate,SalesAmount,RollingMean7,RollingStd7
0,2009-12-01,42708.22,NaN,NaN
1,2009-12-02,52578.19,NaN,NaN
2,2009-12-03,61534.22,NaN,NaN
3,2009-12-04,33686.86,NaN,NaN
4,2009-12-05,9803.05,NaN,NaN
5,2009-12-06,24284.28,NaN,NaN
6,2009-12-07,32423.30,36716.874286,17368.157228
7,2009-12-08,39030.25,36191.450000,17211.625441
8,2009-12-09,30894.06,33093.717143,15651.441643
9,2009-12-10,37706.73,29689.790000,10009.298992


In [62]:
daily_sales.to_csv(
    "rolling_statistics.csv",
    index=False
)

print("Rolling Statistics Saved")

Rolling Statistics Saved


In [25]:
pip install great-expectations

   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ------ --------------------------------- 0.8/4.9 MB 3.7 MB/s eta 0:00:02
   ---------- ----------------------------- 1.3/4.9 MB 3.5 MB/s eta 0:00:02
   --------------------- ------------------ 2.6/4.9 MB 4.0 MB/s eta 0:00:01
   --------------------------- ------------ 3.4/4.9 MB 4.2 MB/s eta 0:00:01
   -------------------------------------- - 4.7/4.9 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 4.9/4.9 MB 4.5 MB/s  0:00:01
   ---------------------------------------- 0.0/676.6 kB ? eta -:--:--
   ---------------------------------------- 676.6/676.6 kB 2.4 MB/s  0:00:00

   ---------------------------------------- 0/5 [tzlocal]
   -------- ------------------------------- 1/5 [tqdm]
   -------- ------------------------------- 1/5 [tqdm]
   -------- ------------------------------- 1/5 [tqdm]
   -------- -------------------------


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\ASJ\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [63]:
import great_expectations as gx
print(gx.__version__)

1.19.0


In [64]:
assert df1["Invoice"].notna().all(), "Invoice column contains null values"

print("✓ Invoice validation passed")

✓ Invoice validation passed


In [65]:
assert df1["Customer ID"].notna().all(), "Customer ID contains null values"

print("✓ Customer ID validation passed")

✓ Customer ID validation passed


In [66]:
negative_qty = df1[df1["Quantity"] < 0]

print("Negative Quantity Rows:", len(negative_qty))
negative_qty.head()

Negative Quantity Rows: 9839


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Customer_Segment,Sales_Channel,Payment_Mode,Order_Priority,SalesAmount
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Regular,Online,UPI,Low,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,Premium,Offline,Card,Medium,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,New,Retail,Cash,Low,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,Regular,Online,Net Banking,High,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Premium,Retail,UPI,Medium,-35.4


In [67]:
df1 = df1[df1["Quantity"] >= 0]

In [68]:
print((df1["Quantity"] < 0).sum())

0


In [69]:
assert (df1["Quantity"] >= 0).all()

print("✓ Quantity validation passed")

✓ Quantity validation passed


In [70]:
print("========== VALIDATION REPORT ==========")
print("Invoice Null Check:", df1["Invoice"].notna().all())
print("Customer ID Check:", df1["Customer ID"].notna().all())
print("Quantity Positive Check:", (df1["Quantity"] >= 0).all())
print("=======================================")

========== VALIDATION REPORT ==========
Invoice Null Check: True
Customer ID Check: True
Quantity Positive Check: True
